In [1]:
#0-1　環境セットアップ　#カグル使用時は不要
!pip install -q numpy==1.26.4 catboost==1.2.7 optuna lightgbm scikit-learn pyarrow

# ランタイム再起動（pip後に1回だけ実行、2回目以降はスキップ）
# import os
# os._exit(0)

In [2]:
#0-2　基本設定

import numpy as np
import pandas as pd
import lightgbm as lgb
import catboost as cb
from catboost import CatBoostClassifier, Pool
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import pickle
import gc
import warnings
import time
import os

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
N_FOLDS = 5
N_OPTUNA_TRIALS = 50  # GPU版: フルトライアル

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


 Home Credit Default Risk — モデリングノートブック（GPU版）

 CPU版からの変更点:
 - CatBoost: task_type="GPU", devices="0"
 - Optuna: フルデータ × 5-Fold × 50トライアル（サンプリング不要）
 - Optuna SQLiteストレージ（セッション切れ時に途中再開可能）
 - boost round増加: LGBM Optuna 3000, CatBoost Optuna 2000, CatBoost本番 5000

 | モデル | CPU版スコア |
 |--------|-----------|
 | LGBM OOF AUC | 0.79271 |
 | CatBoost OOF AUC | 0.78992 |
 | Ensemble OOF AUC | 0.79343 |
 | Private Score | 0.79234 |
 | Public Score | 0.79556 |

In [4]:
#0-3　データ読込

SAVE_DIR = "/content/drive/MyDrive/home_credit_models_gpu"
#SAVE_DIR = "/kaggle/working/home_credit_models_gpu"　#カグル用
os.makedirs(SAVE_DIR, exist_ok=True)

train = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Kaggle/home-credit-default-risk/加工データ/03FE2/train_FE2.zstd.parquet')
test  = pd.read_parquet('/content/drive/MyDrive/Colab Notebooks/Kaggle/home-credit-default-risk/加工データ/03FE2/test_FE2.zstd.parquet')

print(f"train: {train.shape}")
print(f"test:  {test.shape}")
print(f"TARGET分布:\n{train['TARGET'].value_counts(normalize=True)}")

train: (307511, 671)
test:  (48744, 670)
TARGET分布:
TARGET
0.0    0.919271
1.0    0.080729
Name: proportion, dtype: float64


In [5]:
#1-1　下準備（ID・ターゲット分離）

TARGET = "TARGET"
ID_COL = "SK_ID_CURR"

y = train[TARGET].values
test_ids = test[ID_COL].values

drop_cols = [TARGET, ID_COL] if ID_COL in train.columns else [TARGET]
X = train.drop(columns=drop_cols)
X_test = test.drop(columns=[c for c in drop_cols if c in test.columns])

# カラム順序を揃える
X_test = X_test[X.columns]
print(f"特徴量数: {X.shape[1]}")

# --- カテゴリ列の特定 ---
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"カテゴリ列数: {len(cat_cols)}")
if cat_cols:
    print(f"例: {cat_cols[:10]}")

特徴量数: 669
カテゴリ列数: 39
例: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']


 Optuna ハイパーパラメータ探索

 GPU版: フルデータ × 5-Fold × 50トライアル  
 SQLiteストレージでセッション切れ時に途中再開可能  

In [6]:
#2-1　Optuna（LGBM）

def objective_lgbm(trial):
    """LightGBM用Optuna目的関数（フルデータ使用）"""
    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "n_jobs": -1,
        "seed": SEED,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 7),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 15.0),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
    }

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    auc_scores = []

    for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols, free_raw_data=False)
        dval   = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_cols, free_raw_data=False)

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=3000,
            valid_sets=[dval],
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=False),
                lgb.log_evaluation(period=0),
            ],
        )
        preds = model.predict(X_va, num_iteration=model.best_iteration)
        auc_scores.append(roc_auc_score(y_va, preds))

        del dtrain, dval, model
        gc.collect()

    return np.mean(auc_scores)

# %%
print("=" * 60)
print("LightGBM Optuna 探索開始（GPU版: フルデータ × 5-Fold × 50トライアル）")
print("=" * 60)

# SQLiteストレージ: セッション切れ時に途中再開可能
LGBM_DB = os.path.join(SAVE_DIR, "lgbm_optuna.db")
study_lgbm = optuna.create_study(
    direction="maximize",
    study_name="lgbm_hc_gpu",
    storage=f"sqlite:///{LGBM_DB}",
    load_if_exists=True,
)

remaining = N_OPTUNA_TRIALS - len(study_lgbm.trials)
print(f"完了済み: {len(study_lgbm.trials)} / 残り: {remaining}")

if remaining > 0:
    study_lgbm.optimize(objective_lgbm, n_trials=remaining, show_progress_bar=True)

print(f"\nベスト AUC: {study_lgbm.best_value:.5f}")
print(f"ベストパラメータ:")
for k, v in study_lgbm.best_params.items():
    print(f"  {k}: {v}")

LightGBM Optuna 探索開始（GPU版: フルデータ × 5-Fold × 50トライアル）
完了済み: 50 / 残り: 0

ベスト AUC: 0.79242
ベストパラメータ:
  learning_rate: 0.012648663918474872
  num_leaves: 22
  max_depth: 10
  min_child_samples: 92
  reg_alpha: 0.0014782630816190094
  reg_lambda: 0.05326720642915736
  colsample_bytree: 0.34869023688761225
  subsample: 0.8009856822237997
  subsample_freq: 3
  scale_pos_weight: 1.5144113088356903
  min_split_gain: 0.33275419524677236


In [ ]:
'''
============================================================
LightGBM Optuna 探索開始（GPU版: フルデータ × 5-Fold × 50トライアル）
============================================================
完了済み: 44 / 残り: 6
Best trial: 49. Best value: 0.792423: 100%
 6/6 [8:09:26<00:00, 4927.03s/it]

ベスト AUC: 0.79242
ベストパラメータ:
  learning_rate: 0.012648663918474872
  num_leaves: 22
  max_depth: 10
  min_child_samples: 92
  reg_alpha: 0.0014782630816190094
  reg_lambda: 0.05326720642915736
  colsample_bytree: 0.34869023688761225
  subsample: 0.8009856822237997
  subsample_freq: 3
  scale_pos_weight: 1.5144113088356903
  min_split_gain: 0.33275419524677236
  '''

初回は12時間弱でおよそ15トライアルであった。LGBMに拘りがなければGPU環境下ではXGBとCATの組み合わせのほうが早い可能性が高い。  
ただ、懸念点としてXGBのレベルワイズ法とCATの対象木は階層ごとに進めていく点で仕組みが似通っており、アンサンブル効果は薄い可能性がある。  
LGBMのリーフワイズ方式とCATの対称木（Symmetric Tree）方式は木の構築アルゴリズムが異なるため、アンサンブル時にお互いの欠点を補完しやすいと考えられる。

メモ  
ブラウザの設定→パフォーマンスからコラブとカグルを常にアクティブ化し  
CAT(GPU)はカグル、LGBMはコラブで同時並行を行う。

In [ ]:
# #2-2　CatBoost用前処理

# def prepare_catboost_data(df, cat_columns):
#     """CatBoostが受け付ける形にカテゴリ列を変換する。
#     downcast済みデータの型衝突を回避するため、NumPy経由で処理。
#     """
#     df_out = df.copy()
#     for col in cat_columns:
#         arr = df_out[col].values.astype(str)
#         arr = np.where(arr == "nan", "Unknown", arr)
#         df_out[col] = arr
#     return df_out

# X_cat = prepare_catboost_data(X, cat_cols)
# X_test_cat = prepare_catboost_data(X_test, cat_cols)

# cat_indices = [X_cat.columns.get_loc(c) for c in cat_cols]

In [ ]:
# #2-3　Optuna（CatBoost） GPU使用

# def objective_catboost(trial):
#     """CatBoost用Optuna目的関数（GPU + フルデータ使用）"""
#     params = {
#         "loss_function": "Logloss",
#         "eval_metric": "AUC",
#         "task_type": "GPU",
#         "devices": "0", 　　　　#カグル(GPU T4 x2)使用時は"devices": "0:1",
#         "random_seed": SEED,
#         "verbose": 0,
#         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
#         "depth": trial.suggest_int("depth", 4, 10),
#         "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
#         "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
#         "random_strength": trial.suggest_float("random_strength", 0.0, 10.0),
#         "border_count": trial.suggest_int("border_count", 32, 255),
#         "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 15.0),
#         "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 50),
#     }

#     skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
#     auc_scores = []

#     for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X_cat, y)):
#         X_tr = X_cat.iloc[tr_idx]
#         X_va = X_cat.iloc[va_idx]
#         y_tr, y_va = y[tr_idx], y[va_idx]

#         pool_tr = Pool(X_tr, label=y_tr, cat_features=cat_indices)
#         pool_va = Pool(X_va, label=y_va, cat_features=cat_indices)

#         model = CatBoostClassifier(
#             iterations=2000,
#             early_stopping_rounds=100,
#             **params,
#         )
#         model.fit(pool_tr, eval_set=pool_va, verbose=0)

#         preds = model.predict_proba(pool_va)[:, 1]
#         auc_scores.append(roc_auc_score(y_va, preds))

#         del pool_tr, pool_va, model
#         gc.collect()

#     return np.mean(auc_scores)

# # %%
# print("=" * 60)
# print("CatBoost Optuna 探索開始（GPU版: フルデータ × 5-Fold × 50トライアル）")
# print("=" * 60)

# # SQLiteストレージ: セッション切れ時に途中再開可能
# CB_DB = os.path.join(SAVE_DIR, "catboost_optuna.db")
# study_cb = optuna.create_study(
#     direction="maximize",
#     study_name="catboost_hc_gpu",
#     storage=f"sqlite:///{CB_DB}",
#     load_if_exists=True,
# )

# remaining = N_OPTUNA_TRIALS - len(study_cb.trials)
# print(f"完了済み: {len(study_cb.trials)} / 残り: {remaining}")

# if remaining > 0:
#     study_cb.optimize(objective_catboost, n_trials=remaining, show_progress_bar=True)

# print(f"\nベスト AUC: {study_cb.best_value:.5f}")
# print(f"ベストパラメータ:")
# for k, v in study_cb.best_params.items():
#     print(f"  {k}: {v}")

In [ ]:
'''
============================================================
CatBoost Optuna 探索開始（GPU版: フルデータ × 5-Fold × 50トライアル）
============================================================

ベスト AUC: 0.79160
ベストパラメータ:
  learning_rate: 0.03582914737420223
  depth: 5
  l2_leaf_reg: 1.7584954859243032
  bagging_temperature: 0.2324379413937288
  random_strength: 2.5083359358378665
  border_count: 221
  scale_pos_weight: 2.266763421463244
  min_data_in_leaf: 33
  '''

'\n============================================================\nCatBoost Optuna 探索開始（GPU版: フルデータ × 5-Fold × 50トライアル）\n============================================================\n\nベスト AUC: 0.79160\nベストパラメータ:\n  learning_rate: 0.03582914737420223\n  depth: 5\n  l2_leaf_reg: 1.7584954859243032\n  bagging_temperature: 0.2324379413937288\n  random_strength: 2.5083359358378665\n  border_count: 221\n  scale_pos_weight: 2.266763421463244\n  min_data_in_leaf: 33\n  '

In [ ]:
#3-1　LGBM本番学習（最適パラメータ StratifiedKFold）

# best_lgbm_params = {
#     "objective": "binary",
#     "metric": "auc",
#     "boosting_type": "gbdt",
#     "verbosity": -1,
#     "n_jobs": -1,
#     "seed": SEED,
#     **study_lgbm.best_params,
# }

# skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# oof_lgbm = np.zeros(len(X))
# preds_lgbm = np.zeros(len(X_test))
# lgbm_models = []
# lgbm_importances = []

# print("=" * 60)
# print("LightGBM 本番学習")
# print("=" * 60)

# for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
#     t0 = time.time()
#     X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
#     y_tr, y_va = y[tr_idx], y[va_idx]

#     dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols, free_raw_data=False)
#     dval   = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_cols, free_raw_data=False)

#     model = lgb.train(
#         best_lgbm_params,
#         dtrain,
#         num_boost_round=5000,
#         valid_sets=[dval],
#         callbacks=[
#             lgb.early_stopping(stopping_rounds=200, verbose=False),
#             lgb.log_evaluation(period=500),
#         ],
#     )

#     oof_lgbm[va_idx] = model.predict(X_va, num_iteration=model.best_iteration)
#     preds_lgbm += model.predict(X_test, num_iteration=model.best_iteration) / N_FOLDS

#     # 特徴量重要度
#     imp = pd.DataFrame({
#         "feature": X.columns,
#         "importance": model.feature_importance(importance_type="gain"),
#         "fold": fold_idx,
#     })
#     lgbm_importances.append(imp)

#     auc = roc_auc_score(y_va, oof_lgbm[va_idx])
#     elapsed = time.time() - t0
#     print(f"  Fold {fold_idx}: AUC={auc:.5f}  best_iter={model.best_iteration}  ({elapsed:.0f}s)")

#     # --- Fold単位で即保存 & メモリ解放 ---
#     with open(os.path.join(SAVE_DIR, f"lgbm_fold{fold_idx}.pkl"), "wb") as f:
#         pickle.dump(model, f)
#     del dtrain, dval, model
#     gc.collect()
#     print(f"  Fold {fold_idx} 保存・メモリ解放完了")

# oof_auc_lgbm = roc_auc_score(y, oof_lgbm)
# print(f"\n>>> LightGBM OOF AUC: {oof_auc_lgbm:.5f}")

LightGBM 本番学習
[500]	valid_0's auc: 0.777911
[1000]	valid_0's auc: 0.785611
[1500]	valid_0's auc: 0.787892
[2000]	valid_0's auc: 0.788917
[2500]	valid_0's auc: 0.789324
[3000]	valid_0's auc: 0.789865
[3500]	valid_0's auc: 0.790162
  Fold 0: AUC=0.79020  best_iter=3508  (2397s)
  Fold 0 保存・メモリ解放完了
[500]	valid_0's auc: 0.786999
[1000]	valid_0's auc: 0.794886
[1500]	valid_0's auc: 0.797066
[2000]	valid_0's auc: 0.798097
[2500]	valid_0's auc: 0.7986
[3000]	valid_0's auc: 0.798892
  Fold 1: AUC=0.79895  best_iter=2939  (1000s)
  Fold 1 保存・メモリ解放完了
[500]	valid_0's auc: 0.77733
[1000]	valid_0's auc: 0.785145
[1500]	valid_0's auc: 0.787385
[2000]	valid_0's auc: 0.788358
[2500]	valid_0's auc: 0.788678
[3000]	valid_0's auc: 0.789186
  Fold 2: AUC=0.78924  best_iter=3234  (1151s)
  Fold 2 保存・メモリ解放完了
[500]	valid_0's auc: 0.783479
[1000]	valid_0's auc: 0.790583
[1500]	valid_0's auc: 0.792662
[2000]	valid_0's auc: 0.793853
[2500]	valid_0's auc: 0.794641
[3000]	valid_0's auc: 0.794928
  Fold 3: AUC=0.7

In [ ]:
'''
============================================================
LightGBM 本番学習
============================================================
[500]	valid_0's auc: 0.777911
[1000]	valid_0's auc: 0.785611
[1500]	valid_0's auc: 0.787892
[2000]	valid_0's auc: 0.788917
[2500]	valid_0's auc: 0.789324
[3000]	valid_0's auc: 0.789865
[3500]	valid_0's auc: 0.790162
  Fold 0: AUC=0.79020  best_iter=3508  (2397s)
  Fold 0 保存・メモリ解放完了
[500]	valid_0's auc: 0.786999
[1000]	valid_0's auc: 0.794886
[1500]	valid_0's auc: 0.797066
[2000]	valid_0's auc: 0.798097
[2500]	valid_0's auc: 0.7986
[3000]	valid_0's auc: 0.798892
  Fold 1: AUC=0.79895  best_iter=2939  (1000s)
  Fold 1 保存・メモリ解放完了
[500]	valid_0's auc: 0.77733
[1000]	valid_0's auc: 0.785145
[1500]	valid_0's auc: 0.787385
[2000]	valid_0's auc: 0.788358
[2500]	valid_0's auc: 0.788678
[3000]	valid_0's auc: 0.789186
  Fold 2: AUC=0.78924  best_iter=3234  (1151s)
  Fold 2 保存・メモリ解放完了
[500]	valid_0's auc: 0.783479
[1000]	valid_0's auc: 0.790583
[1500]	valid_0's auc: 0.792662
[2000]	valid_0's auc: 0.793853
[2500]	valid_0's auc: 0.794641
[3000]	valid_0's auc: 0.794928
  Fold 3: AUC=0.79499  best_iter=3199  (1134s)
  Fold 3 保存・メモリ解放完了
[500]	valid_0's auc: 0.777132
[1000]	valid_0's auc: 0.785388
[1500]	valid_0's auc: 0.787827
[2000]	valid_0's auc: 0.789124
[2500]	valid_0's auc: 0.789936
[3000]	valid_0's auc: 0.790232
  Fold 4: AUC=0.79028  best_iter=2838  (1036s)
  Fold 4 保存・メモリ解放完了

>>> LightGBM OOF AUC: 0.79271
'''

In [8]:
train.shape

(307511, 671)

AUCは30トライアルと同値であったため、次回以降はShape(307511, 671)以下のデータ量であれば30以下で実施

In [ ]:
#3-2　LGBM結果保存

# pd.DataFrame({
#     "SK_ID_CURR": train["SK_ID_CURR"].values,
#     "oof_lgbm": oof_lgbm,
#     "TARGET": y,
# }).to_parquet(os.path.join(SAVE_DIR, "oof_lgbm.parquet"), index=False)

# pd.DataFrame({
#     "SK_ID_CURR": test_ids,
#     "pred_lgbm": preds_lgbm,
# }).to_parquet(os.path.join(SAVE_DIR, "pred_lgbm.parquet"), index=False)

# imp_df = pd.concat(lgbm_importances)
# imp_df.to_parquet(os.path.join(SAVE_DIR, "lgbm_importance.parquet"), index=False)

# print(f"LGBM保存完了: {oof_auc_lgbm:.5f}")

LGBM保存完了: 0.79271


In [ ]:
#3-3　CatBoost本番学習（最適パラメータ StratifiedKFold）GPU使用

# best_cb_params = {
#     "loss_function": "Logloss",
#     "eval_metric": "AUC",
#     "task_type": "GPU",
#     "devices": "0",　　　　　#カグル(GPU T4 x2)使用時は"devices": "0:1",
#     "random_seed": SEED,
#     "verbose": 0,
#     **study_cb.best_params,
# }

# skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# oof_cb = np.zeros(len(X_cat))
# preds_cb = np.zeros(len(X_test_cat))

# print("=" * 60)
# print("CatBoost 本番学習（GPU）")
# print("=" * 60)

# for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X_cat, y)):
#     t0 = time.time()
#     X_tr = X_cat.iloc[tr_idx]
#     X_va = X_cat.iloc[va_idx]
#     y_tr, y_va = y[tr_idx], y[va_idx]

#     pool_tr = Pool(X_tr, label=y_tr, cat_features=cat_indices)
#     pool_va = Pool(X_va, label=y_va, cat_features=cat_indices)

#     model = CatBoostClassifier(
#         iterations=5000,
#         early_stopping_rounds=200,
#         **best_cb_params,
#     )
#     model.fit(pool_tr, eval_set=pool_va, verbose=500)

#     oof_cb[va_idx] = model.predict_proba(pool_va)[:, 1]
#     preds_cb += model.predict_proba(Pool(X_test_cat, cat_features=cat_indices))[:, 1] / N_FOLDS

#     auc = roc_auc_score(y_va, oof_cb[va_idx])
#     elapsed = time.time() - t0
#     print(f"  Fold {fold_idx}: AUC={auc:.5f}  best_iter={model.best_iteration_}  ({elapsed:.0f}s)")

#     # 寄与度取得
#     imp = pd.DataFrame({
#         "feature": X_cat.columns,
#         "importance": model.get_feature_importance(),
#         "fold": fold_idx,
#     })
#     imp.to_parquet(os.path.join(SAVE_DIR, f"catboost_imp_fold{fold_idx}.parquet"), index=False)

#     # --- Fold単位で即保存 & メモリ解放 ---
#     model.save_model(os.path.join(SAVE_DIR, f"catboost_fold{fold_idx}.cbm"))
#     del pool_tr, pool_va, model
#     gc.collect()
#     print(f"  Fold {fold_idx} 保存・メモリ解放完了")

# oof_auc_cb = roc_auc_score(y, oof_cb)
# print(f"\n>>> CatBoost OOF AUC: {oof_auc_cb:.5f}")

In [ ]:
'''
============================================================
CatBoost 本番学習（GPU）
============================================================
Default metric period is 5 because AUC is/are not implemented for GPU
0:	test: 0.6984224	best: 0.6984224 (0)	total: 75.3ms	remaining: 6m 16s
500:	test: 0.7777388	best: 0.7777388 (500)	total: 26.4s	remaining: 3m 56s
1000:	test: 0.7843770	best: 0.7843770 (1000)	total: 52.1s	remaining: 3m 28s
1500:	test: 0.7868985	best: 0.7868985 (1500)	total: 1m 17s	remaining: 3m 1s
2000:	test: 0.7881425	best: 0.7881484 (1993)	total: 1m 43s	remaining: 2m 34s
2500:	test: 0.7889167	best: 0.7889197 (2495)	total: 2m 8s	remaining: 2m 8s
3000:	test: 0.7893147	best: 0.7893453 (2921)	total: 2m 33s	remaining: 1m 42s
3500:	test: 0.7896293	best: 0.7896336 (3493)	total: 2m 59s	remaining: 1m 16s
4000:	test: 0.7895388	best: 0.7897590 (3822)	total: 3m 24s	remaining: 51.1s
bestTest = 0.7897590399
bestIteration = 3822
Shrink model to first 3823 iterations.
  Fold 0: AUC=0.78976  best_iter=3822  (217s)
  Fold 0 保存・メモリ解放完了
Default metric period is 5 because AUC is/are not implemented for GPU
0:	test: 0.7046861	best: 0.7046861 (0)	total: 75.5ms	remaining: 6m 17s
500:	test: 0.7885267	best: 0.7885267 (500)	total: 26.6s	remaining: 3m 59s
1000:	test: 0.7952191	best: 0.7952191 (1000)	total: 52.4s	remaining: 3m 29s
1500:	test: 0.7973818	best: 0.7973936 (1494)	total: 1m 17s	remaining: 3m 1s
2000:	test: 0.7985132	best: 0.7985132 (2000)	total: 1m 43s	remaining: 2m 34s
bestTest = 0.7985858619
bestIteration = 2043
Shrink model to first 2044 iterations.
  Fold 1: AUC=0.79859  best_iter=2043  (127s)
  Fold 1 保存・メモリ解放完了
Default metric period is 5 because AUC is/are not implemented for GPU
0:	test: 0.7005515	best: 0.7005515 (0)	total: 76.8ms	remaining: 6m 23s
500:	test: 0.7795980	best: 0.7795980 (500)	total: 26.6s	remaining: 3m 58s
1000:	test: 0.7857895	best: 0.7857895 (1000)	total: 52.3s	remaining: 3m 28s
1500:	test: 0.7875869	best: 0.7875869 (1500)	total: 1m 17s	remaining: 3m 1s
2000:	test: 0.7883300	best: 0.7883770 (1955)	total: 1m 43s	remaining: 2m 34s
bestTest = 0.7886369824
bestIteration = 2148
Shrink model to first 2149 iterations.
  Fold 2: AUC=0.78864  best_iter=2148  (132s)
  Fold 2 保存・メモリ解放完了
Default metric period is 5 because AUC is/are not implemented for GPU
0:	test: 0.7083119	best: 0.7083119 (0)	total: 74.4ms	remaining: 6m 11s
500:	test: 0.7856133	best: 0.7856133 (500)	total: 26.5s	remaining: 3m 58s
1000:	test: 0.7917169	best: 0.7917169 (1000)	total: 52.1s	remaining: 3m 28s
1500:	test: 0.7936497	best: 0.7936497 (1500)	total: 1m 17s	remaining: 3m 1s
2000:	test: 0.7946398	best: 0.7946398 (2000)	total: 1m 43s	remaining: 2m 34s
2500:	test: 0.7949770	best: 0.7950510 (2411)	total: 2m 8s	remaining: 2m 8s
bestTest = 0.7953101397
bestIteration = 2630
Shrink model to first 2631 iterations.
  Fold 3: AUC=0.79531  best_iter=2630  (157s)
  Fold 3 保存・メモリ解放完了
Default metric period is 5 because AUC is/are not implemented for GPU
0:	test: 0.6961010	best: 0.6961010 (0)	total: 76ms	remaining: 6m 19s
500:	test: 0.7780768	best: 0.7780768 (500)	total: 26.6s	remaining: 3m 58s
1000:	test: 0.7851602	best: 0.7851602 (1000)	total: 52.5s	remaining: 3m 29s
1500:	test: 0.7875427	best: 0.7875430 (1496)	total: 1m 17s	remaining: 3m 1s
2000:	test: 0.7888510	best: 0.7888862 (1993)	total: 1m 43s	remaining: 2m 35s
2500:	test: 0.7892912	best: 0.7892974 (2489)	total: 2m 9s	remaining: 2m 9s
3000:	test: 0.7897183	best: 0.7897426 (2925)	total: 2m 34s	remaining: 1m 43s
bestTest = 0.789789021
bestIteration = 3143
Shrink model to first 3144 iterations.
  Fold 4: AUC=0.78979  best_iter=3143  (184s)
  Fold 4 保存・メモリ解放完了

>>> CatBoost OOF AUC: 0.79233
'''

In [ ]:
#3-4　CatBoost結果保存

# pd.DataFrame({
#     "SK_ID_CURR": train["SK_ID_CURR"].values,
#     "oof_catboost": oof_cb,
#     "TARGET": y,
# }).to_parquet(os.path.join(SAVE_DIR, "oof_catboost.parquet"), index=False)

# pd.DataFrame({
#     "SK_ID_CURR": test_ids,
#     "pred_catboost": preds_cb,
# }).to_parquet(os.path.join(SAVE_DIR, "pred_catboost.parquet"), index=False)

# print(f"CatBoost保存完了: {oof_auc_cb:.5f}")

| Fold | CPU版(trial 10) | GPU版(trial 50) | 差分 |
|:---|:---|:---|:---|
| 0 | 0.78656 | 0.78976 | +0.0032 |
| 1 | 0.79520 | 0.79859 | +0.0034 |
| 2 | 0.78698 | 0.78864 | +0.0017 |
| 3 | 0.79460 | 0.79531 | +0.0007 |
| 4 | 0.78647 | 0.78979 | +0.0033 |
| **OOF** | **0.78992** | **0.79233** | **+0.0024** |

In [9]:
#3-5　LGBM結果復元

SAVE_DIR = "/content/drive/MyDrive/home_credit_models_gpu"

oof_lgbm_df = pd.read_parquet(os.path.join(SAVE_DIR, "oof_lgbm.parquet"))
oof_lgbm = oof_lgbm_df["oof_lgbm"].values
y = oof_lgbm_df["TARGET"].values

pred_lgbm_df = pd.read_parquet(os.path.join(SAVE_DIR, "pred_lgbm.parquet"))
preds_lgbm = pred_lgbm_df["pred_lgbm"].values
test_ids = pred_lgbm_df["SK_ID_CURR"].values

oof_auc_lgbm = roc_auc_score(y, oof_lgbm)
print(f"LGBM OOF AUC (復元): {oof_auc_lgbm:.5f}")

LGBM OOF AUC (復元): 0.79271


In [12]:
#3-6　CAT結果復元

# CatBoost GPU版の復元
SAVE_DIR_CAT = "/content/drive/MyDrive/home_credit_models_gpu/home_credit_models_gpu_CAT"

oof_cb_df = pd.read_parquet(os.path.join(SAVE_DIR_CAT, "oof_catboost.parquet"))
oof_cb = oof_cb_df["oof_catboost"].values

pred_cb_df = pd.read_parquet(os.path.join(SAVE_DIR_CAT, "pred_catboost.parquet"))
preds_cb = pred_cb_df["pred_catboost"].values

oof_auc_cb = roc_auc_score(y, oof_cb)
print(f"CatBoost OOF AUC (GPU版復元): {oof_auc_cb:.5f}")

CatBoost OOF AUC (GPU版復元): 0.79233


In [13]:
#4-1　アンサンブル（重み調整）

def find_best_weight(oof_a, oof_b, y_true, steps=101):
    """OOFスコアで最適なブレンド重みをグリッドサーチする。"""
    best_w, best_auc = 0.0, 0.0
    results = []
    for w in np.linspace(0, 1, steps):
        blended = w * oof_a + (1 - w) * oof_b
        auc = roc_auc_score(y_true, blended)
        results.append((w, auc))
        if auc > best_auc:
            best_w, best_auc = w, auc
    return best_w, best_auc, results

w_lgbm, ensemble_auc, weight_results = find_best_weight(oof_lgbm, oof_cb, y)

print("=" * 60)
print("アンサンブル結果")
print("=" * 60)
print(f"最適重み: LGBM={w_lgbm:.2f}, CatBoost={1 - w_lgbm:.2f}")
print(f"Ensemble OOF AUC: {ensemble_auc:.5f}")
print(f"  (LGBM単体: {oof_auc_lgbm:.5f})")
print(f"  (CatBoost単体: {oof_auc_cb:.5f})")

アンサンブル結果
最適重み: LGBM=0.59, CatBoost=0.41
Ensemble OOF AUC: 0.79411
  (LGBM単体: 0.79271)
  (CatBoost単体: 0.79233)


In [16]:
#4-2　テスト予測・提出ファイル作成

preds_ensemble = w_lgbm * preds_lgbm + (1 - w_lgbm) * preds_cb

sub = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: preds_ensemble,
})
sub.to_csv(os.path.join(SAVE_DIR, "submission.csv"), index=False)
print(f"\nsubmission.csv 作成完了: {sub.shape}")
print(sub.head())


submission.csv 作成完了: (48744, 2)
   SK_ID_CURR    TARGET
0      100001  0.084753
1      100005  0.175176
2      100013  0.050114
3      100028  0.041099
4      100038  0.239969


In [15]:
#5-1　全結果保存

# OOF（アンサンブル込み）
oof_all = pd.DataFrame({
    "SK_ID_CURR": train["SK_ID_CURR"].values,
    "oof_lgbm": oof_lgbm,
    "oof_catboost": oof_cb,
    "oof_ensemble": w_lgbm * oof_lgbm + (1 - w_lgbm) * oof_cb,
    "TARGET": y,
})
oof_all.to_parquet(os.path.join(SAVE_DIR, "oof_all.parquet"), index=False)

# テスト予測（アンサンブル込み）
test_all = pd.DataFrame({
    "SK_ID_CURR": test_ids,
    "pred_lgbm": preds_lgbm,
    "pred_catboost": preds_cb,
    "pred_ensemble": preds_ensemble,
})
test_all.to_parquet(os.path.join(SAVE_DIR, "test_all.parquet"), index=False)

print(f"全結果保存完了: {os.listdir(SAVE_DIR)}")

全結果保存完了: ['home_credit_models_gpu_CAT', 'catboost_info', 'lgbm_optuna.db', 'lgbm_fold0.pkl', 'lgbm_fold1.pkl', 'lgbm_fold2.pkl', 'lgbm_fold3.pkl', 'lgbm_fold4.pkl', 'oof_lgbm.parquet', 'pred_lgbm.parquet', 'lgbm_importance.parquet', 'submission.csv', 'oof_all.parquet', 'test_all.parquet']


まとめ

CPU版との比較
| 項目 | CPU版 | GPU版 | 差分 |
|:---|:---|:---|:---|
| LGBM Optuna | 25%サンプル × 3-Fold × 30trial | フルデータ × 5-Fold × 50trial | 75% × 2-Fold × trial20 |
| CatBoost Optuna | 25%サンプル × 3-Fold × 10trial | フルデータ × 5-Fold × 50trial | 75% × 2-Fold × trial40 |
| LGBM OOF AUC | 0.79271 | 0.79271 | ±0 |
| CatBoost OOF AUC | 0.78992 | 0.79233 | +0.00241 |
| Ensemble OOF AUC | 0.79343 | 0.79411 | +0.00068 |
| Private Score |	0.79234 | 0.79293 | +0.00059 |
| Public Score | 0.79556 | 0.79608 | +0.00052 |
| Private Rank | 1423/7180(上位20％以内) | 1032/7180(上位15％以内) | ▲391 |
| Public Rank | 2771/7180(上位40％以内) | 2534/7180(上位40％以内) | ▲237 |


【AUC】  
CAT自体はトライアル10→40で0.24%の改善となり、多少の恩恵を得られたが、LGBMは30→50でスコアは変わらなかった。  
また、CATの改善幅もアンサンブルすることで小さくなり、最終的なスコアとしてはプライベートもパブリックも0.05%程度の改善であった。
次回以降はデータサイズと時間帯効率を意識したトライアルで実施を行う。  
  
また、更なるスコア向上を目指す場合としては以下の方法が考えられるが、当該コンペではスコア面の追求は一旦区切りとつけて、次のステップ(結果の分析)に移りたい。  


*   サブテーブルのFE  
ほとんどのサブテーブル集約時において、一括のFE(min.max,ave,std)で行っており、内容の属性等の検討は見送っているため、ここのFE追加が一番大きく期待できる。
*   ノイズとなるカラムの検討・削除  
SHAP等で寄与度上位陣のカラム分析を行い、ノイズ等の削除を行うことで精度向上できると思われるが、具体的な判断が困難となる可能性がある。  
*   XGBoost,NN追加及びアンサンブル
*   Stacking  
※（OOFの予測値自体を特徴量として学習させて2回目のモデルに学習させる手法）





【RANK】  
実務面においてはあまり意味のない指標だと考えられるが、プライベートとパブリックに差が大きく現れた点は興味深く、他者との比較において、恒常的には強いが直近予測には弱いモデルであることが判明した。